# EXP-011 infer-only S+C+A fold0

鏃犺缁冿細鍔犺浇 Dataset 鏉冮噸 + 鎸傝浇 test 涓婃帹鐞?+ mean銆傞伩鍏嶈瘎鍒嗛噸璺戣秴鏃躲€俓nDataset: `ziyiming/rsna-exp011-e5-sca-weights`
- rsna_v3 `2f268c17cb958d50d50a320a9bfdac96a05178b888d25b2cda8116fcf979a774`
- n_clips=2 鈫?channels=6


In [ ]:
# 鎻愪氦妯″紡:绂佺綉(enable_internet:false),torch 2.2.0+cu118 + nvidia-cu11 渚濊禆鍏ㄩ儴绂荤嚎瀹夎
# 鍧?:Kaggle 鎸傝浇鎶婃枃浠跺悕涓?'+' 鍘绘帀(2.2.0cu118=闈炴硶 PEP440)鈫?澶嶅埗涓哄悎娉曞悕
# 鍧?:torch 2.2.0+cu118 鐨?CUDA 搴撳湪 nvidia-*-cu11 渚濊禆鍖呬腑(涓嶅湪 torch wheel 鍐?,
#     鍙 torch 浼氱己 libcudart(v8/v9 瀹炶瘉)鈫?蹇呴』鏄惧紡瀹夎鍏ㄩ儴 13 涓?wheel
# 鍧?:/kaggle/working 浼氳鏀堕泦涓?kernel 杈撳嚭 鈫?瑁呭畬绔嬪嵆鍒犻櫎
import subprocess, sys, os, glob, shutil
wh = sorted(glob.glob('/kaggle/input/**/*.whl', recursive=True))
print('鎸傝浇 wheel 鍏?, len(wh), '涓?)
for w in wh: print('  ', os.path.basename(w))
assert len(wh) >= 13, 'FAIL: 绂荤嚎 wheel 涓嶈冻(搴?13: torch+torchvision+11 nvidia-cu11)'
fixed = []
for w in wh:
    base = os.path.basename(w)
    dst = os.path.join('/kaggle/working', base.replace('cu118', '+cu118', 1) if 'cu118' in base else base)
    shutil.copy(w, dst)
    fixed.append(dst)
print('瀹夎 wheel:', [os.path.basename(f) for f in fixed])
subprocess.check_call([sys.executable,"-m","pip","uninstall","-y","-q","torch","torchvision","torchaudio"])
# 鏄惧紡瀹夎鍏ㄩ儴 wheel(--no-deps --no-index: 鏃犵増鏈В鏋?鏃犺仈缃?
subprocess.check_call([sys.executable,"-m","pip","install","-q","--no-deps","--no-index"] + fixed)
for f in fixed: os.remove(f)
import numpy, torch, torchvision, sklearn, pandas
print("torch", torch.__version__, "cuda", torch.version.cuda, "| numpy", numpy.__version__)
assert torch.cuda.is_available()
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
import base64, os, hashlib
parts=[]; parts.append("...")
path="/kaggle/working/rsna_v3.py"
open(path,"w",encoding="utf-8").write(base64.b64decode(parts[0]).decode("utf-8"))
h=hashlib.sha256(open(path,"rb").read()).hexdigest()
print("rsna_v3",h); assert h=="2f268c17cb958d50d50a320a9bfdac96a05178b888d25b2cda8116fcf979a774"
print("v3 OK")


In [ ]:
import os, glob
import numpy as np, pandas as pd, torch
import rsna_v3 as V

root, out = V.resolve_paths()
print("root", root)
test_df = pd.read_csv(os.path.join(root, "test.csv"))
test_series = pd.read_csv(os.path.join(root, "test_series.csv"))
dcm_test = os.path.join(root, "test_series")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device", device, "test", len(test_df))

# dataset layout is flat: .../rsna-exp011-e5-sca-weights/Sagittal/model.pt
hits = glob.glob("/kaggle/input/**/model.pt", recursive=True)
print("model.pt hits:")
for h in hits:
    print(" ", h)
assert hits, "no model.pt under /kaggle/input"

# map plane -> path
plane_paths = {}
for h in hits:
    parent = os.path.basename(os.path.dirname(h))  # Sagittal / Coronal / Axial / fold0
    # also support fold0/Plane/model.pt
    if parent in ("Sagittal", "Coronal", "Axial"):
        plane_paths[parent] = h
    else:
        grand = os.path.basename(os.path.dirname(os.path.dirname(h)))
        if grand in ("Sagittal", "Coronal", "Axial"):
            plane_paths[grand] = h
        elif parent == "fold0":
            # dirname is plane
            plane = os.path.basename(os.path.dirname(h))
            # actually .../fold0/Plane/model.pt -> parent is Plane
            pass
print("plane_paths", plane_paths)
for p in ["Sagittal", "Coronal", "Axial"]:
    assert p in plane_paths, f"missing {p} weights, have {list(plane_paths)}"

channels = 2 * V.CLIP
preds = []
for plane in ["Sagittal", "Coronal", "Axial"]:
    wpath = plane_paths[plane]
    model = V.KneeModel(num_labels=len(V.LABELS), channels=channels,
                        pretrained=False, offline_paths=None).to(device)
    sd = torch.load(wpath, map_location=device)
    model.load_state_dict(sd)
    model.eval()
    with torch.no_grad():
        pr = V.predict_test(model, test_df, test_series, dcm_test, device, plane, n_clips=2)
    print(plane, "pred", pr.shape, float(pr.mean()))
    preds.append(np.asarray(pr, dtype=np.float64))

fused = np.mean(np.stack(preds, 0), 0)
df = pd.DataFrame(fused, columns=V.LABELS)
df.insert(0, "StudyInstanceUID", test_df["StudyInstanceUID"].tolist())
df.to_csv("submission.csv", index=False)
print(df.to_string())
print("wrote submission.csv", df.shape)
